In [89]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/winterbreeze/fifa19eda/fifa_eda.csv


In [90]:
df=pd.read_csv("/kaggle/input/datasets/winterbreeze/fifa19eda/fifa_eda.csv")

In [91]:
df.head()

,ID,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Preferred Foot,International Reputation,Skill Moves,Position,Joined,Contract Valid Until,Height,Weight,Release Clause
0,158023,L. Messi,31,Argentina,94,94,FC Barcelona,110500.0,565.0,Left,5.0,4.0,RF,2004,2021-01-01,5.583333,159.0,226500.0
1,20801,Cristiano Ronaldo,33,Portugal,94,94,Juventus,77000.0,405.0,Right,5.0,5.0,ST,2018,2022-01-01,6.166667,183.0,127100.0
2,190871,Neymar Jr,26,Brazil,92,93,Paris Saint-Germain,118500.0,290.0,Right,5.0,5.0,LW,2017,2022-01-01,5.750000,150.0,228100.0
3,193080,De Gea,27,Spain,91,93,Manchester United,72000.0,260.0,Right,4.0,1.0,GK,2011,2020-01-01,6.333333,168.0,138600.0
4,192985,K. De Bruyne,27,Belgium,91,92,Manchester City,102000.0,355.0,Right,4.0,4.0,RCM,2015,2023-01-01,5.916667,154.0,196400.0


In [92]:
df.shape

(18207, 18)

In [93]:
df.isnull().sum()

ID                            0
Name                          0
Age                           0
Nationality                   0
Overall                       0
Potential                     0
Club                        241
Value                       252
Wage                          0
Preferred Foot                0
International Reputation     48
Skill Moves                  48
Position                      0
Joined                        0
Contract Valid Until        289
Height                        0
Weight                        0
Release Clause                0
dtype: int64

In [94]:
(df['Club']=='nan').value_counts()

Club
False    18207
Name: count, dtype: int64

In [95]:
df=df.dropna()

In [96]:
df.shape

(17907, 18)

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17907 entries, 0 to 18206
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        17907 non-null  int64  
 1   Name                      17907 non-null  object 
 2   Age                       17907 non-null  int64  
 3   Nationality               17907 non-null  object 
 4   Overall                   17907 non-null  int64  
 5   Potential                 17907 non-null  int64  
 6   Club                      17907 non-null  object 
 7   Value                     17907 non-null  float64
 8   Wage                      17907 non-null  float64
 9   Preferred Foot            17907 non-null  object 
 10  International Reputation  17907 non-null  float64
 11  Skill Moves               17907 non-null  float64
 12  Position                  17907 non-null  object 
 13  Joined                    17907 non-null  int64  
 14  Contract Va

In [98]:
cat_cols=df.select_dtypes(include='object')
num_cols=df.select_dtypes(exclude='object')

In [99]:
for i in cat_cols:
    print(i,"--->",df[i].nunique())

Name ---> 16913
Nationality ---> 163
Club ---> 651
Preferred Foot ---> 2
Position ---> 27
Contract Valid Until ---> 35


In [100]:
df.columns

Index(['ID', 'Name', 'Age', 'Nationality', 'Overall', 'Potential', 'Club',
       'Value', 'Wage', 'Preferred Foot', 'International Reputation',
       'Skill Moves', 'Position', 'Joined', 'Contract Valid Until', 'Height',
       'Weight', 'Release Clause'],
      dtype='object')

In [101]:
df.drop(columns=['ID','Name'],axis=1,inplace=True)

In [102]:
df.shape

(17907, 16)

In [103]:
df['Nationality'].unique()

array(['Argentina', 'Portugal', 'Brazil', 'Spain', 'Belgium', 'Croatia',
       'Uruguay', 'Slovenia', 'Poland', 'Germany', 'France', 'England',
       'Italy', 'Egypt', 'Colombia', 'Denmark', 'Gabon', 'Wales',
       'Senegal', 'Costa Rica', 'Slovakia', 'Netherlands',
       'Bosnia Herzegovina', 'Morocco', 'Serbia', 'Algeria', 'Austria',
       'Greece', 'Chile', 'Sweden', 'Korea Republic', 'Finland', 'Guinea',
       'Montenegro', 'Armenia', 'Switzerland', 'Norway', 'Czech Republic',
       'Scotland', 'Ghana', 'Central African Rep.', 'DR Congo',
       'Ivory Coast', 'Russia', 'Ukraine', 'Iceland', 'Mexico', 'Jamaica',
       'Albania', 'Venezuela', 'Japan', 'Turkey', 'Ecuador', 'Paraguay',
       'Mali', 'Nigeria', 'Cameroon', 'Dominican Republic', 'Israel',
       'Kenya', 'Hungary', 'Republic of Ireland', 'Romania',
       'United States', 'Cape Verde', 'Australia', 'Peru', 'Togo',
       'Syria', 'Zimbabwe', 'Angola', 'Burkina Faso', 'Iran', 'Estonia',
       'Tunisia', 'Equato

In [104]:
import pycountry
import pycountry_convert as pc

def get_continent(nationality):

    try:
        country = pycountry.countries.lookup(nationality)
        alpha2 = country.alpha_2

        continent_code = pc.country_alpha2_to_continent_code(alpha2)

        continent_name = pc.convert_continent_code_to_continent_name(continent_code)

        return continent_name

    except:
        return 'Unknown'

In [105]:
df['Continent'] = df['Nationality'].apply(get_continent)

In [106]:
df.columns

Index(['Age', 'Nationality', 'Overall', 'Potential', 'Club', 'Value', 'Wage',
       'Preferred Foot', 'International Reputation', 'Skill Moves', 'Position',
       'Joined', 'Contract Valid Until', 'Height', 'Weight', 'Release Clause',
       'Continent'],
      dtype='object')

In [107]:
df['Contract Valid Until'].unique()

array(['2021-01-01', '2022-01-01', '2020-01-01', '2023-01-01',
       '2019-01-01', '2024-01-01', '2019-06-30', '2025-01-01',
       '2026-01-01', '2018-12-31', '2018-01-01', '2020-05-31',
       '2020-06-30', '2019-05-31', '2019-12-31', '2019-06-01',
       '2019-01-04', '2019-01-31', '2019-01-07', '2019-01-02',
       '2019-01-06', '2019-10-14', '2019-01-03', '2019-05-04',
       '2019-01-12', '2019-01-25', '2019-01-18', '2019-12-01',
       '2018-11-30', '2020-02-27', '2019-01-05', '2019-01-15',
       '2019-01-30', '2019-01-11', '2019-01-20'], dtype=object)

In [108]:
df['Contract_End_Year']=df['Contract Valid Until'].str.split("-").str[0]

In [109]:
df.drop(columns='Contract Valid Until',axis=1,inplace=True)

In [110]:
df['Joined'].unique()

array([2004, 2018, 2017, 2011, 2015, 2012, 2014, 2005, 2010, 2016, 2008,
       2013, 2007, 2009, 2002, 2003, 2006, 2001, 1991, 1998, 2000, 1999])

In [111]:
df['Joined']=df['Joined'].astype('int64')

In [112]:
print(df['Joined'].dtype)
print(df['Contract_End_Year'].dtype)

int64
object


In [113]:
df['Contract_End_Year']=df['Contract_End_Year'].astype('int64')
print(df['Contract_End_Year'].dtype)

int64


In [114]:
df['Retained']=df['Contract_End_Year']-df['Joined']

In [115]:
df.head()

,Age,Nationality,Overall,Potential,Club,Value,Wage,Preferred Foot,International Reputation,Skill Moves,Position,Joined,Height,Weight,Release Clause,Continent,Contract_End_Year,Retained
0,31,Argentina,94,94,FC Barcelona,110500.0,565.0,Left,5.0,4.0,RF,2004,5.583333,159.0,226500.0,South America,2021,17
1,33,Portugal,94,94,Juventus,77000.0,405.0,Right,5.0,5.0,ST,2018,6.166667,183.0,127100.0,Europe,2022,4
2,26,Brazil,92,93,Paris Saint-Germain,118500.0,290.0,Right,5.0,5.0,LW,2017,5.750000,150.0,228100.0,South America,2022,5
3,27,Spain,91,93,Manchester United,72000.0,260.0,Right,4.0,1.0,GK,2011,6.333333,168.0,138600.0,Europe,2020,9
4,27,Belgium,91,92,Manchester City,102000.0,355.0,Right,4.0,4.0,RCM,2015,5.916667,154.0,196400.0,Europe,2023,8


In [116]:
df['Age']=df['Age'].astype('int64')

In [117]:
df['Age_Group'] = pd.cut(
    pd.to_numeric(df['Age'], errors='coerce'),
    bins=[15, 20, 25, 30, 35, 40, 45],
    labels=['15-19', '20-24', '25-29', '30-34', '35-39', '40-44'],
    right=False
)

In [118]:
df.drop(columns='Age',axis=1,inplace=True)

In [119]:
df.head()

,Nationality,Overall,Potential,Club,Value,Wage,Preferred Foot,International Reputation,Skill Moves,Position,Joined,Height,Weight,Release Clause,Continent,Contract_End_Year,Retained,Age_Group
0,Argentina,94,94,FC Barcelona,110500.0,565.0,Left,5.0,4.0,RF,2004,5.583333,159.0,226500.0,South America,2021,17,30-34
1,Portugal,94,94,Juventus,77000.0,405.0,Right,5.0,5.0,ST,2018,6.166667,183.0,127100.0,Europe,2022,4,30-34
2,Brazil,92,93,Paris Saint-Germain,118500.0,290.0,Right,5.0,5.0,LW,2017,5.750000,150.0,228100.0,South America,2022,5,25-29
3,Spain,91,93,Manchester United,72000.0,260.0,Right,4.0,1.0,GK,2011,6.333333,168.0,138600.0,Europe,2020,9,25-29
4,Belgium,91,92,Manchester City,102000.0,355.0,Right,4.0,4.0,RCM,2015,5.916667,154.0,196400.0,Europe,2023,8,25-29


In [121]:
df.columns

Index(['Nationality', 'Overall', 'Potential', 'Club', 'Value', 'Wage',
       'Preferred Foot', 'International Reputation', 'Skill Moves', 'Position',
       'Joined', 'Height', 'Weight', 'Release Clause', 'Continent',
       'Contract_End_Year', 'Retained', 'Age_Group'],
      dtype='object')